# Positional Embeddings (Encoding word encodings)

In [13]:
import torch
import tiktoken
from torch.utils.data import DataLoader, Dataset

### For the learning purpose we are using the dimension size of 256 which is smaller than the actual dimension size used by GPT-3 which was 12,288 dimensions.
### And the vocabulary size used is 50257 tokens (which is the vocab size of GPT-2)

In [14]:
vocab_size = 50257
output_dims = 256

token_embedding_layer = torch.nn.Embedding(vocab_size, output_dims)

## Datasets & Data loaders

In [15]:
class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, context_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenization
        encoded_text = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

        # Using the sliding window approach to chunk the book content using squences of context_length
        for i in range(0, len(encoded_text)-context_length, stride):
            input_chunk = encoded_text[i:i+context_length]
            target_chunk = encoded_text[i+1:i+context_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]

In [16]:
def create_dataloader_v1(text, batch_size=4, context_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset using the GPTDatasetV1
    dataset = GPTDatasetV1(text, tokenizer, context_length, stride)

    # Create dataloader using the DataLoader from pytorch
    dataloader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

## Initializing the DataLoader

In [17]:
with open("harrypotter.txt", "r", encoding="utf-8") as harryPotterBook:
    raw_text = harryPotterBook.read()

### We will be using the batch_size of 8 along with the context_length of 4

In [18]:
context_length = 4
dataloader = create_dataloader_v1(raw_text, batch_size=8, context_length=context_length, stride=4, shuffle=False)

data_iter = iter(dataloader)
inputs, targets = next(data_iter)

In [19]:
print(f"Token ID's:\n{inputs}")
print(f"\nInput Shape: {inputs.shape}")

Token ID's:
tensor([[18308, 14179,   290,   262],
        [30467,   338,  8026,   220],
        [  198,   198, 41481, 16329],
        [  220,   198,   198, 10970],
        [16494,    56, 19494,   406],
        [ 3824,  1961,   220,   198],
        [  198,  5246,    13,   290],
        [ 9074,    13,   360,  1834]])

Input Shape: torch.Size([8, 4])


## Important Step: Now we will get the token embeddings of each of these token ids in the inputs from the look up table which is the token_embedding_layer which we created above

In [20]:
token_embeddings = token_embedding_layer(inputs)
print(token_embeddings.shape)

torch.Size([8, 4, 256])


### Now we got a tensor of 8X4X256, where 8 is the batch_size, 4 is the context_length and 256 is the output_dims

## Creating another layer for the Positional Encodings 

In [22]:
positional_embedding_layer = torch.nn.Embedding(context_length, output_dims)

In [23]:
positional_embeddings = positional_embedding_layer(torch.arange(context_length))
print(positional_embeddings.shape)

torch.Size([4, 256])


## Now add the positional_embeddings (4X256) directly to the token_embeddings (8X4X256) in order to get the input embeddings. Although these two matrices have different dimensions but they can still be added together and this is possible because of the Python Broadcasting

In [24]:
input_embeddings = token_embeddings + positional_embeddings
print(input_embeddings.shape)

torch.Size([8, 4, 256])


In [26]:
print(input_embeddings)

tensor([[[ 1.3550,  0.0293,  1.1903,  ...,  1.0006, -0.5498, -0.9982],
         [ 1.6647,  0.3531, -0.0669,  ..., -1.6122,  2.0498, -2.9850],
         [ 0.3965, -2.2188, -0.0087,  ..., -0.9386,  1.1973,  2.6098],
         [ 0.6876, -1.2988, -1.8904,  ...,  0.2121, -1.2114, -4.2327]],

        [[ 1.3085, -1.6037, -1.4745,  ...,  0.6978, -0.9755,  0.0610],
         [ 1.9917,  1.7582,  0.9101,  ..., -3.7161,  2.3284, -2.3026],
         [ 1.5320, -1.5458,  0.4910,  ...,  0.8006,  0.8933,  0.3839],
         [-0.4155, -0.2837, -1.1944,  ...,  0.4794, -1.5846, -3.0005]],

        [[ 2.0535, -0.8564, -1.5267,  ...,  0.0325,  1.0209,  0.8239],
         [ 1.2511,  0.0973, -0.7358,  ..., -3.4703,  2.1574, -0.1317],
         [ 1.1054,  0.3537,  1.6721,  ..., -2.5221,  0.1176,  1.2887],
         [ 0.5931, -0.0380, -1.8312,  ...,  0.9843, -2.2535, -3.5503]],

        ...,

        [[ 2.6808, -0.0469,  0.7420,  ...,  2.7239,  0.5560, -1.6876],
         [ 1.6067,  1.5736, -1.3848,  ..., -2.7339,  0.53

# Now these input_embeddings can be fedded to the LLMs for training